In [ ]:
#pip install paddleocr paddlepaddle
#pip install numpy==1.25.2
#pip install --upgrade PyMuPDF==1.25.1
#pip install pdf2image==1.16.3
#apt-get install -y poppler-utils

In [2]:
import fitz
import re
import csv
from paddleocr import PaddleOCR
from pdf2image import convert_from_path

# ---------------- Config ----------------
pdf_path = "IT415-ITElec4(CyberSec)-T1-_APPROVED_syllabus.pdf"
co_csv = "course_outcomes.csv"
po_csv = "program_outcomes.csv"

ocr = PaddleOCR(use_textline_orientation=True, lang='en')


# ---------------- Helper ----------------
def normalize_text(s):
    """Cleans and normalizes extracted text."""
    s = re.sub(r"\s+", " ", s).strip()
    s = re.sub(r"(?:\b[IiEeDd]\b\s*)+$", "", s)
    return s.strip(" .;:-")


def ocr_extract_text(pdf_path):
    """OCR text extraction using PaddleOCR."""
    print("🧠 Running PaddleOCR on PDF pages...")
    images = convert_from_path(pdf_path, dpi=300)
    all_text = ""
    for i, img in enumerate(images, 1):
        result = ocr.ocr(img, cls=True)
        for line in result[0]:
            all_text += line[1][0] + "\n"
        print(f"✅ Page {i}: OCR extracted {len(result[0])} lines")
    return all_text


# ---------------- Extract Course Outcomes ----------------
def extract_course_outcomes(pdf_path):
    """Extract Course Outcomes (COs) while avoiding headers/footers and PO contamination, stopping at first period."""
    import fitz, re

    def normalize_text(s):
        # Remove multiple spaces, strip edges, remove trailing punctuation
        s = re.sub(r"\s+", " ", s).strip()
        return s.strip(" .;:-")

    text = ""
    try:
        with fitz.open(pdf_path) as doc:
            for page in doc:
                page_text = page.get_text()
                # Remove likely headers/footers and document metadata
                page_text = re.sub(
                    r"Document Code No\..*|Rev\. No\..*|Page No\..*|Effective Date.*|FM-USTP-ACAD-\d+|00|\b\d{2}\.\d{2}\.\d{2}\b|\d+\s*of\s*\d+",
                    "",
                    page_text,
                    flags=re.I,
                )
                text += page_text + "\n"
    except Exception:
        text = ocr_extract_text(pdf_path)

    # Extract CO section only
    match = re.search(
        r"(Course Outcomes.*?)(Program Outcomes|Mapping|III\.|Assessment|Grading|Allotted Time|$)",
        text,
        re.I | re.S,
    )
    co_section = match.group(1) if match else text

    # Clean extra spaces
    co_section = re.sub(r"\s{2,}", " ", co_section).strip()

    # Extract COs: CO1, CO2, ...
    co_pattern = r"\b(CO\d+)\s*[:\-–]?\s*(.*?)(?=\bCO\d+\s*[:\-–]|Program Outcomes|III\.|Allotted Time|$)"
    co_matches = re.findall(co_pattern, co_section, flags=re.I | re.S)

    co_rows = []
    for co, desc in co_matches:
        # Prevent PO contamination
        desc = re.split(r"\b[a-o]:", desc, flags=re.I)[0]
        desc = normalize_text(desc)
        # Stop at first proper sentence-ending period
        first_sentence = re.split(r"\.\s", desc)
        if first_sentence:
            desc = first_sentence[0].strip() + "."
        if desc:
            co_rows.append({"CO": co.upper(), "CO_Description": desc})

    return co_rows


# ---------------- Extract Program Outcomes ----------------
def extract_program_outcomes(pdf_path):
    po_blocks = []
    found_po = False

    with fitz.open(pdf_path) as doc:
        for page in doc:
            # ✅ Extract only LEFT-SIDE text (ignore right table)
            blocks = [b for b in page.get_text("blocks") if b[0] < 220]
            text = "\n".join(b[4] for b in blocks)

            if not found_po and re.search(r"Program Outcomes", text, re.I):
                found_po = True
                text = re.split(r"Program Outcomes\s*:", text, flags=re.I)[-1]

            if found_po:
                po_blocks.append(text)
                if re.search(r"(Allotted Time|Week|Course Outcomes|IV\.|Course Requirements|V\.)", text, re.I):
                    found_po = False

    po_text = " ".join(po_blocks)
    po_text = po_text.encode("latin1", errors="ignore").decode("utf-8", errors="ignore")

    # --- Clean metadata ---
    clean_patterns = [
        r"Document Code No\..*",
        r"FM-USTP-ACAD-01.*",
        r"USTP-ACAD.*",
        r"Rev\. No\..*",
        r"Page No\..*",
        r"Effective Date.*",
        r"\b\d+\s*of\s*\d+\b.*",
        r"UNIVERSITY OF SCIENCE AND TECHNOLOGY.*",
        r"\b\d{2}\.\d{2}\.\d{2}\b",  # 03.17.25
        r"\b00\b",
    ]
    for pat in clean_patterns:
        po_text = re.sub(pat, "", po_text, flags=re.I)

    po_text = re.sub(r"-\s*\n\s*", "", po_text)
    po_text = re.sub(r"\s{2,}", " ", po_text).strip()

    # --- Extract POs (a–o) ---
    po_pattern = r"\b([a-o]):\s*(.*?)(?=\s*[a-o]:|$)"
    pos = []
    letter_to_po = {chr(97 + i): f"PO{i+1}" for i in range(15)}

    for code, desc in re.findall(po_pattern, po_text, re.DOTALL):
        desc = re.sub(r"\s+", " ", desc).strip()
        mapped_code = letter_to_po.get(code.lower(), code)
        pos.append({"PO": mapped_code, "PO_Description": desc})

    return pos


# ---------------- Save CSV ----------------
def save_csv(filename, rows, fieldnames):
    with open(filename, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)
    print(f"💾 Saved {len(rows)} rows to {filename}")


# ---------------- Run Extraction ----------------
cos = extract_course_outcomes(pdf_path)
pos = extract_program_outcomes(pdf_path)

save_csv(co_csv, cos, ["CO", "CO_Description"])
save_csv(po_csv, pos, ["PO", "PO_Description"])

print("✅ Extraction completed successfully!")


c:\Users\Dave\AppData\Local\Programs\Python\Python313\Lib\site-packages\paddle\utils\cpp_extension\extension_utils.py:718: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Creating model: ('PP-LCNet_x1_0_doc_ori', None)
Using official model (PP-LCNet_x1_0_doc_ori), the model files will be automatically downloaded and saved in `C:\Users\Dave\.paddlex\official_models\PP-LCNet_x1_0_doc_ori`.


Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

README.md: 0.00B [00:00, ?B/s]

inference.json: 0.00B [00:00, ?B/s]

inference.yml:   0%|          | 0.00/766 [00:00<?, ?B/s]

...

README.md: 0.00B [00:00, ?B/s]

inference.yml: 0.00B [00:00, ?B/s]

inference.json: 0.00B [00:00, ?B/s]

inference.pdiparams:   0%|          | 0.00/7.77M [00:00<?, ?B/s]

💾 Saved 4 rows to course_outcomes.csv
💾 Saved 15 rows to program_outcomes.csv
✅ Extraction completed successfully!
